In [7]:
# ==============================
# IMPORTS
# ==============================

import os
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import librosa
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, average_precision_score
from sklearn.preprocessing import label_binarize, StandardScaler

# ==============================
# DEVICE
# ==============================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# IMAGE TRANSFORM
# ==============================

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ==============================
# AUDIO FEATURE
# ==============================

def extract_audio_features(file_path):
    signal, sr = librosa.load(file_path, sr=22050)
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=40)
    mfcc = np.mean(mfcc.T, axis=0)

    if len(mfcc) < 256:
        mfcc = np.pad(mfcc, (0, 256 - len(mfcc)))
    else:
        mfcc = mfcc[:256]

    return torch.tensor(mfcc).float().to(device)

# ==============================
# MODELS
# ==============================

class VisionTransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        self.vit.heads = nn.Linear(self.vit.heads.head.in_features, 128)

    def forward(self, x):
        return self.vit(x)

class AudioModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

    def forward(self, x):
        return self.net(x)

class FusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU()
        )

    def forward(self, v, a):
        return self.fc(torch.cat((v, a), dim=1))

# ==============================
# LOAD DATASET PATHS
# ==============================

def load_dataset_paths(dataset_path):
    img_paths, aud_paths, labels = [], [], []
    class_names = sorted(os.listdir(dataset_path))

    for label, cls in enumerate(class_names):
        folder = os.path.join(dataset_path, cls)

        imgs = [f for f in os.listdir(folder) if f.endswith(".jpg")]
        auds = [f for f in os.listdir(folder) if f.endswith((".wav", ".mp3"))]

        if len(auds) == 0:
            continue

        for img_file in imgs:
            img_path = os.path.join(folder, img_file)

            if os.path.getsize(img_path) < 5000:
                continue

            aud_file = np.random.choice(auds)
            aud_path = os.path.join(folder, aud_file)

            img_paths.append(img_path)
            aud_paths.append(aud_path)
            labels.append(label)

    return img_paths, aud_paths, labels, class_names

# ==============================
# FEATURE EXTRACTION (FIXED)
# ==============================

def extract_features_from_paths(img_paths, aud_paths, labels, vit, audio_model, fusion):
    features = []
    valid_labels = []

    vit.eval()
    audio_model.eval()
    fusion.eval()

    with torch.no_grad():
        for img_p, aud_p, label in zip(img_paths, aud_paths, labels):
            try:
                img = Image.open(img_p).convert("RGB")
                img = image_transform(img).unsqueeze(0).to(device)

                aud = extract_audio_features(aud_p)

                v = vit(img)
                a = audio_model(aud).unsqueeze(0)
                f = fusion(v, a)

                features.append(f.cpu().numpy().flatten())
                valid_labels.append(label)

            except:
                continue

    return np.array(features), np.array(valid_labels)

# ==============================
# mAP
# ==============================

def compute_map(y_test, y_pred, num_classes):
    y_test_bin = label_binarize(y_test, classes=list(range(num_classes)))
    y_pred_bin = label_binarize(y_pred, classes=list(range(num_classes)))

    scores = []
    for i in range(num_classes):
        scores.append(average_precision_score(y_test_bin[:, i], y_pred_bin[:, i]))

    return np.mean(scores)

# ==============================
# MAIN
# ==============================

def main():

    dataset_path = "D://bird project//datas"

    print("Loading dataset paths...")
    img_paths, aud_paths, labels, class_names = load_dataset_paths(dataset_path)

    print("Total samples:", len(labels))

    # ==============================
    # SPLIT BEFORE FEATURE EXTRACTION
    # ==============================

    X_train_img, X_test_img, X_train_aud, X_test_aud, y_train, y_test = train_test_split(
        img_paths, aud_paths, labels,
        test_size=0.2,
        stratify=labels,
        random_state=42
    )

    vit = VisionTransformerModel().to(device)
    audio_model = AudioModel().to(device)
    fusion = FusionModel().to(device)

    # ==============================
    # SAFE CACHE CHECK
    # ==============================

    if (os.path.exists("train_features.npy") and
        os.path.exists("test_features.npy") and
        os.path.exists("y_train.npy") and
        os.path.exists("y_test.npy")):

        print("Loading saved features...")
        train_features = np.load("train_features.npy")
        test_features = np.load("test_features.npy")
        y_train = np.load("y_train.npy")
        y_test = np.load("y_test.npy")

    else:
        print("Cache not found. Extracting features...")

        train_features, y_train = extract_features_from_paths(
            X_train_img, X_train_aud, y_train, vit, audio_model, fusion
        )

        test_features, y_test = extract_features_from_paths(
            X_test_img, X_test_aud, y_test, vit, audio_model, fusion
        )

        np.save("train_features.npy", train_features)
        np.save("test_features.npy", test_features)
        np.save("y_train.npy", y_train)
        np.save("y_test.npy", y_test)

    # ==============================
    # NORMALIZATION
    # ==============================

    scaler = StandardScaler()
    train_features = scaler.fit_transform(train_features)
    test_features = scaler.transform(test_features)

    # ==============================
    # MODEL
    # ==============================

    classifier = RandomForestClassifier(
        n_estimators=500,
        max_depth=30,
        random_state=42
    )

    classifier.fit(train_features, y_train)

    predictions = classifier.predict(test_features)

    # ==============================
    # OUTPUT
    # ==============================

    print("\n===== SAMPLE PREDICTIONS =====\n")

    for i in range(min(10, len(predictions))):
        print(f"Predicted: {class_names[predictions[i]]}")
        print(f"Actual   : {class_names[y_test[i]]}")
        print("-----------------------------")

    accuracy = accuracy_score(y_test, predictions) * 100
    map_score = compute_map(y_test, predictions, len(class_names)) * 100

    print("\n===== PERFORMANCE METRICS =====\n")
    print(f"Accuracy               : {accuracy:.2f}%")
    print(f"Mean Average Precision: {map_score:.2f}%")

    joblib.dump(classifier, "model.pkl")
    print("\nModel saved successfully!")

# ==============================
# RUN
# ==============================

if __name__ == "__main__":
    main()

Loading dataset paths...
Total samples: 3728
Cache not found. Extracting features...


C:\Users\gokil\AppData\Local\Temp\ipykernel_31052\737809723.py:40: UserWarning: PySoundFile failed. Trying audioread instead.
  signal, sr = librosa.load(file_path, sr=22050)
C:\Users\gokil\anaconda3\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



===== SAMPLE PREDICTIONS =====

Predicted: common myna
Actual   : common myna
-----------------------------
Predicted: pigeon
Actual   : pigeon
-----------------------------
Predicted: Sparrow
Actual   : Sparrow
-----------------------------
Predicted: Parrot
Actual   : Parrot
-----------------------------
Predicted: Parrot
Actual   : Parrot
-----------------------------
Predicted: pigeon
Actual   : pigeon
-----------------------------
Predicted: Woodpecker
Actual   : Woodpecker
-----------------------------
Predicted: crow
Actual   : crow
-----------------------------
Predicted: crow
Actual   : crow
-----------------------------
Predicted: eagle
Actual   : eagle
-----------------------------

===== PERFORMANCE METRICS =====

Accuracy               : 98.08%
Mean Average Precision: 95.89%

Model saved successfully!


In [31]:
import joblib

model = joblib.load("model.pkl")   # load existing model
joblib.dump(model, "model_compressed.pkl", compress=9)

from IPython.display import FileLink

FileLink("model_compressed.pkl")

C:\Users\gokil\model_compressed.pkl

In [35]:
import re

with open("bird_project.ipynb", "r", encoding="utf-8") as f:
    code = f.read()

# remove comments
code = re.sub(r"#.*", "", code)

# remove extra blank lines
code = "\n".join([line for line in code.splitlines() if line.strip()])

with open("app_min.py", "w", encoding="utf-8") as f:
    f.write(code)

print("Minified file created!")

Minified file created!
